In [ ]:
import os
import json
import torch
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars
from safetensors.torch import load_file

from llm.pretrained_weight_loader import PretrainedQweenModel
from inference import Qwen3InferenceEngine
from tokenizer.qween_3_tokenizer import Qwen3Tokenizer

In [2]:
# inference.py

# Ensure 'tokenizer-base.json' is accessible
tokenizer_path = "tokenizer.json"
if not os.path.exists(tokenizer_path):
    raise FileNotFoundError(f"Please provide '{tokenizer_path}' inside the root folder.")
    
print("Loading Tokenizer...")
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

print("Loading Model and Weights...")
model, _ = PretrainedQweenModel.from_pretrained(config_path="config.json", weight_path="model.safetensors")

# Initialize our new Inference Engine
engine = Qwen3InferenceEngine(model=model, tokenizer=tokenizer)
print(f"Engine ready on device: {engine.device}")

Loading Tokenizer...
Loading Model and Weights...
Building configuration from config.json...
Initializing empty architecture...
Loading weights from model.safetensors...
Mapping and loading weights...


Injecting Transformer Weights: 100%|████████████| 28/28 [00:08<00:00,  3.24it/s]


Pretrained weights loaded successfully!

Engine ready on device: cpu


In [4]:
# Run tests
test_prompt = "Deep learning is a subset of"

engine.stream_to_console(test_prompt, max_new_tokens=40, temperature=0.7, top_p=0.9)
print(f"n")

engine.stream_to_console(test_prompt, max_new_tokens=40, temperature=0.7, top_p=0.9)
print(f"\n")


--- Streaming Output (Temp=0.7, Top-p=0.9) ---
Deep learning is a subset of artificial intelligence that has gained significant attention in recent years. The field is characterized by the use of algorithms and models that are highly efficient and capable of handling complex tasks. As the demand for these models grows

--- Generation Complete ---
n

--- Streaming Output (Temp=0.7, Top-p=0.9) ---
Deep learning is a subset of artificial intelligence that is designed to make humans perform tasks that require intelligence. It is built with the help of algorithms, data, and neural networks. The field of AI has been growing rapidly, and the

--- Generation Complete ---




# reasoning.py

In [ ]:
import os
import math
import torch

from inference import Qwen3InferenceEngine
from tokenizer.qween_3_tokenizer import Qwen3Tokenizer
from llm.pretrained_weight_loader import PretrainedQweenModel
from llm.rl.reasoning import ConversationalReasoner, ReasoningMetrics

# Ensure 'tokenizer.json' is accessible
tokenizer_path = "tokenizer.json"
if not os.path.exists(tokenizer_path):
    raise FileNotFoundError(f"Please provide '{tokenizer_path}' inside the root folder.")

# setup Exodia Pieces
print("Loading Core Architecture...")
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)
model, _ = PretrainedQweenModel.from_pretrained("config.json", "model.safetensors")
base_engine = Qwen3InferenceEngine(model=model, tokenizer=tokenizer)

# instantiate our new Object-Oriented Handlers
agent = ConversationalReasoner(base_engine)
metrics = ReasoningMetrics(base_engine)

Loading Core Architecture...
Building configuration from config.json...
Initializing empty architecture...
Loading weights from model.safetensors...
Mapping and loading weights...


Injecting Transformer Weights: 100%|████████████| 28/28 [00:21<00:00,  1.32it/s]


Pretrained weights loaded successfully!



In [ ]:
# test the Self-Refinement Engine
user_question = "Explain quantum computing in one short paragraph for a 10-year-old."

print("\n--- Starting Agentic Refinement Loop ---")
results = agent.refine(
    user_input=user_question,
    iterations=2,
    max_response_tokens=1500,
    max_critique_tokens=1000,
    verbose=True
)

print("\nFINAL DELIVERABLE:")
print(results["final_response"])


--- Starting Agentic Refinement Loop ---
Generating initial draft...
Critiquing draft 1/2...
Writing final revision 1/2...

[Editing Step 1/2]
Original Draft: Maybe add some examples of real-life applications? Also, make sure to use simple and clear language.
Answer:
Quantum computers are like super smart machines that can do calculations faster than regular ones! They work by using tiny particles called qubits instead of just bits (which we usually call "yes" or "no"). These qubits can be both 0 and 1 at the same time—like being half asleep and full awake—but only if they're placed correctly. This allows them to solve problems much quicker. Real-world uses include things like finding hidden patterns in data quickly or optimizing complex systems efficiently.

**Example:** Imagine you have a lot of tasks to complete, say building something complicated with many different options. A regular computer might take forever
Critique: The example was too technical. We should simplify it even m